Install Required Libraries

In [ ]:
!pip install langchain langchain-community langchain-core
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install groq
!pip install langgraph
!pip install langchain-community
!pip install langchain-text-splitters

Import Libraries

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from groq import Groq

Set Groq API Key

In [ ]:
os.environ["GROQ_API_KEY"] = "your API key"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

Upload PDF

In [ ]:
from google.colab import files
uploaded = files.upload()

Load the PDF

In [ ]:
loader = PyPDFLoader("sample.pdf")
documents = loader.load()

print("Total pages:", len(documents))

Chunk the Document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # size of each chunk
    chunk_overlap=50       # overlap for better context
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("Sample chunk:\n", chunks[0].page_content)

Create Embeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Store in ChromaDB

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print("Vector DB created successfully")

Create Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Test Retrieval

In [ ]:
query = "Can I use the card for online transactions?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

Create LLM Function

In [ ]:
def generate_answer(query, context):
    prompt = f"""
You are a helpful customer support assistant.

Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="llama3-8b-8192",  # fast + free in Groq
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

Combine Retrieval + LLM

In [ ]:
def rag_pipeline(query):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    answer = generate_answer(query, context)

    return answer

Test Chatbot

In [ ]:
def generate_answer(query, context):
    prompt = f"""
You are a helpful customer support assistant.

Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
!pip install langgraph

Define State

In [ ]:
from typing import TypedDict

class GraphState(TypedDict):
    query: str
    context: str
    answer: str
    confidence: float

Processing Node

In [ ]:
def processing_node(state):
    query = state["query"]

    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    answer = generate_answer(query, context)

    # Simple confidence logic
    if "I don't know" in answer:
        confidence = 0.2
    else:
        confidence = 0.9

    return {
        "context": context,
        "answer": answer,
        "confidence": confidence
    }

Decision Function

In [ ]:
def route_decision(state):
    if state["confidence"] < 0.5:
        return "hitl"
    else:
        return "output"

Output Node

In [ ]:
def output_node(state):
    print("\n✅ Final Answer:\n", state["answer"])
    return state

HITL Node

In [ ]:
def hitl_node(state):
    print("\n⚠️ Escalating to Human Support...")
    print("Query:", state["query"])

    human_response = input("👨‍💻 Enter human response: ")

    return {
        "answer": human_response,
        "confidence": 1.0
    }

Build LangGraph

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

builder.add_node("process", processing_node)
builder.add_node("output", output_node)
builder.add_node("hitl", hitl_node)

builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    route_decision,
    {
        "output": "output",
        "hitl": "hitl"
    }
)

builder.set_finish_point("output")

graph = builder.compile()

Run the Graph

In [ ]:
query = "What is my account balance?"

graph.invoke({"query": query})